# Étape 2 — Baseline et métriques

**Projet 5 · LoRA Zellige** — à exécuter sur Google Colab avec un GPU T4.

Ce notebook ne fait **aucun entraînement**. Il construit le point de
comparaison « avant » et vérifie que les mesures fonctionnent.

Ordre volontaire : on fige les prompts et on code les métriques **avant**
d'entraîner quoi que ce soit. Sinon on choisit, même sans le vouloir, les
prompts et les mesures qui avantagent son modèle.

> Menu **Exécution → Modifier le type d'exécution → GPU T4**


## 1. Vérifier le GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA disponible :', torch.cuda.is_available())


## 2. Installer les dépendances

Torch est déjà présent sur Colab, on n'installe que le reste.


In [ ]:
!pip -q install 'diffusers>=0.27' 'peft>=0.10' 'transformers>=4.38' \
    accelerate safetensors lpips pyyaml
print('installation terminée')


## 3. Récupérer le dépôt


In [ ]:
!git clone -q https://github.com/HatimTj/lora-zellige-patrimoine.git /content/projet || (cd /content/projet && git pull -q)
%cd /content/projet
!ls


## 4. Reconstruire le jeu de données

Les images ne sont **pas** dans Git (consigne du sujet). Les fichiers
texte versionnés suffisent à les reconstruire à l'identique :

| Fichier | Rôle |
|---|---|
| `data/licenses.csv` | nom de fichier → titre Commons, auteur, licence |
| `data/selection.txt` | le tri manuel des 37 images |
| `data/compositions.csv` | les descriptions écrites à la main |
| `data/preparation.json` | la séparation entraînement / généralisation |

On utilise `--reconstruire`, qui redemande à Commons **les fichiers
exacts** enregistrés dans `licenses.csv`, au lieu de relancer une
recherche. Deux raisons :

1. **Reproductibilité.** En mode recherche, le nom `zlj_042.jpg`
   dépend de l'ordre des résultats. Si une requête échoue ou si
   Commons publie de nouvelles images, la numérotation se décale et
   `selection.txt` ne désigne plus les bonnes images.
2. **Débit.** Une poignée d'appels au lieu de plusieurs dizaines :
   les adresses IP de Colab sont partagées et Wikimedia les limite
   vite (erreur HTTP 429).

Comptez environ 1 minute.


In [ ]:
# Les && arretent la chaine des qu'une etape echoue,
# au lieu de continuer sur un jeu de donnees vide.
!python src/collect_wikimedia.py --reconstruire \
  && python src/prepare_data.py \
  && python src/legender.py


In [ ]:
# Contrôle : 29 images d'entraînement, 8 de généralisation
!ls data/processed/train/*.jpg | wc -l
!ls data/processed/holdout/*.jpg | wc -l
!head -3 data/processed/train/metadata.jsonl


## 5. Auto-test des métriques

Avant de mesurer un vrai modèle, on vérifie que les mesures se
comportent correctement sur des cas évidents : une image rouge unie
doit obtenir un CLIPScore élevé face au texte « a solid red image »,
et la diversité LPIPS de trois images identiques doit être nulle.

Le test vérifie aussi que les vecteurs d'image et de texte ont la
**même dimension** (512). Sans la projection de CLIP, l'image sort en
768 et le calcul est impossible.

> L'avertissement `You are sending unauthenticated requests to the HF
> Hub` est sans gravité : il signale seulement un débit de
> téléchargement bridé.

Une métrique qu'on n'a pas testée est une métrique à laquelle on ne
peut pas croire.


In [ ]:
!python src/metrics.py


## 6. Générer la baseline

Stable Diffusion 1.5 **sans aucune LoRA**, sur les 10 prompts figés × 4
graines = 40 images. Premier lancement : ~4 Go de modèle à télécharger.


In [ ]:
!python src/generate.py --sortie results/baseline


## 7. Évaluer la baseline

Ajoute une ligne dans `results/results.csv`. C'est la ligne de référence :
toutes les LoRA seront comparées à elle.


In [ ]:
!python src/evaluate.py --dossier results/baseline --nom baseline


## 8. Regarder les images

Les prompts contiennent `zlj_style`, un token que le modèle ne connaît pas
encore. Il va donc l'ignorer et produire ce que lui inspirent les autres
mots. **C'est exactement le comportement attendu** — c'est le « avant ».


In [ ]:
from PIL import Image
import yaml, matplotlib.pyplot as plt

prompts = yaml.safe_load(open('configs/eval_prompts.yaml'))['prompts']
graine = 1234

fig, axes = plt.subplots(2, 5, figsize=(20, 8.6))
for ax, p in zip(axes.ravel(), prompts):
    ax.imshow(Image.open(f"results/baseline/{p['id']}_s{graine}.png"))
    ax.set_title(f"{p['id']} · {p['categorie']}", fontsize=10)
    ax.axis('off')
plt.suptitle('Baseline SD 1.5 — avant LoRA (graine 1234)', fontsize=14)
plt.tight_layout(); plt.show()


## 9. Lire les résultats


In [ ]:
import pandas as pd
pd.read_csv('results/results.csv').T


### Comment lire ces chiffres

| Colonne | Sens | Attendu pour la baseline |
|---|---|---|
| `style_dino_moy` | ressemblance au zellige | **faible** — le modèle ne le connaît pas |
| `prompt_domaine` | respect du texte sur les prompts zellige | moyen |
| `prompt_hors_domaine` | respect du texte hors sujet | **élevé** — c'est son point fort |
| `memorisation_max_dino` | copie-t-il une image d'entraînement ? | **très faible** — il ne les a jamais vues |
| `diversite_lpips` | les 4 graines donnent-elles 4 images différentes ? | élevée |

Ces valeurs sont le **point de départ**. À l'Étape 4, `style_dino_moy` doit
monter nettement, et il faudra surveiller si `prompt_hors_domaine` chute —
ce serait le signe que la LoRA a abîmé le modèle de base.


## 10. Sauvegarder les résultats

Les 40 images sont trop lourdes pour Git. On ne renvoie que les fichiers
de mesures, qui pèsent quelques kilo-octets.


In [ ]:
!git config user.name 'HatimTj'
!git config user.email 'hatimtajimi1@gmail.com'
!git add results/results.csv results/baseline/manifeste.json \
         results/baseline/evaluation.json
!git commit -m 'etape2: baseline SD 1.5 et mesures de reference'

# Pour pousser depuis Colab il faut un jeton GitHub personnel :
#   !git push https://VOTRE_JETON@github.com/HatimTj/lora-zellige-patrimoine.git main
# Sinon, telechargez les trois fichiers depuis le panneau de gauche
# et commitez-les depuis votre Mac.
